# EBI APIs for some International Space Station dust

<img src="iss-dust.png" width="640">

_LLM hallucination of dust collection..._

---

## 0. Find the dataset


There are various ways to find interesting datasets at EBI... one of the best entry points is the "EBI Search" site.
(By the way, this too has an API! But often you'd be exploring this sort of thing by hand or with an LLM.)

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">
    
1. ☑️ Go to the EBI homepage: https://www.ebi.ac.uk/
2. ☑️ In the search box at the top, look for `international space station vacuum cleaner`
3. ☑️ Check out the top hit... as perhaps expected, it is a metagenomics study (in the MGnify resource) about the microbial communities in a vacuum cleaner.... on the ISS 🛰️
4. ☑️ Click on it. You should end up at https://www.ebi.ac.uk/metagenomics/studies/MGYS00005116/overview 
</div>

## 1. Using the MGnify API

### 1.1 Let's explore this first service's API.

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">

1. ☑️ If you like, click around the MGnify pages for the study a bit. If you're into microbial environments it might even be interesting!
2. ☑️ On that Study Overview page there is a URL for an API endpoint... copy it.
3. ☑️ In the cell below, complete the `curl ... ` command to call that API endpoint:

</div>

In [ ]:
%%bash
curl -s "https://...."

**Try to complete the cell above. If you get too stuck, unfold the cell below to see a solution.**

In [ ]:
%%bash
curl -s "https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00005116"

### 1.2 Readable command line

That is ~unreadable unless you're from the Matrix. Let's use a handy terminal tool called `jq` to make it readable...

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">
    
☑️ To do so, **pipe** the output from `curl` into the command `jq`. 
</div>

(i.e. `curl ... | jq `)

In [ ]:
%%bash
# curl MGnify API into jq

**Try to complete the cell above. If you get too stuck, unfold the cell below to see a solution.**

In [ ]:
%%bash
curl -s "https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00005116" | jq

### 1.3 Selecting some API data on the command line

`jq` is more than a pretty-printer... it can do pretty clever traversal of the `JSON` response. For example, to find the Study Title:

In [ ]:
%%bash
curl -s "https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00005116" | jq '.title'

Easy! That selected the part of the JSON tree at the _JSON path_ of `.title`. 
It is also possible (but more fiddly) to deal with the arrays in the response. E.g., you might have noticed an array of `downloads`, and some of those mention different MGnify pipelines versions in their `download_group`. 
<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">
    
☑️ Try to use the `jq` documentation (e.g. [the array iterator](https://jqlang.org/manual/#array-object-value-iterator), the [select](https://jqlang.org/manual/#select) and [contains](https://jqlang.org/manual/#contains) functions, and the [pipe |](https://jqlang.org/manual/#pipe) operator) to construct a one-liner terminal command that will get the JSON from that Study endpoont, and spit out the `url` of each `download` file with `v6` (the latest MGnify pipeline) contained in its `download_group`...

</div>

(This is a bit harder!)

In [ ]:
%%bash
curl -s "https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00005116" | jq ...

**Try to complete the cell above. If you get too stuck, unfold the cell below to see a solution.**

In [ ]:
%%bash
curl -s "https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00005116" | jq '.downloads[] | select(.download_group | contains("v6")) | .url'

### 1.4 Downloading data on the command line

To actually download one of these files, we can pass each `url` in turn to another invocation of `curl`.
<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">
    
☑️ To do so, add ` | xargs -n1 curl -O` to the end of your previous command (assuming you had it spitting out 1 URL per line)
</div>

In [ ]:
%%bash

curl ...

**Try to complete the cell above. If you get too stuck, unfold the cell below to see a solution.**

In [ ]:
%%bash
curl -s "https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00005116" | jq '.downloads[] | select(.download_group | contains("v6")) | .url' | xargs -n1 curl -O


You should now have some TSV files downloaded:

In [ ]:
!ls *.tsv

In [ ]:
!head *.tsv

### 1.5 Using a client library

Unless you're a unix aficionado, you might be thinking now that this is all too fiddly, and you're probably correct.

There are two good directions to go in:
1. Use a "browsable API", i.e. one where a user interface is provided, and you can just click around a bit like a website. The one for MGnify can be found by going to https://www.ebi.ac.uk/metagenomics/api/v2/ in the browser. You can see all the endpoints and click the "Try it out" button to make requests similarly to what we did the `curl`. You will also see there, that it explains the SCHEMA of each request (and response).
2. That SCHEMA can also help build other clients... for example, one in Python, and that's what we will look at next: fetching data from the API directly in Python.

Let's install [MGnipy](https://mgnipy.mgnify.org/) - the Python client for MGnify's API.

In [ ]:
!pip install mgnipy

In [ ]:
from mgnipy import MGnipy

# Create the main client, with default configuration
mg = MGnipy()

# See available endpoints
mg.list_resources()

#### 1.5.1 Find the studies

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">
    
☑️  Use the same text search query we started with on the EBI website to look for dusty ISS studies, but this time via MGnipy (e.g. see [the docs](https://mgnipy.mgnify.org/#search-resources-with-a-mgnipy-mgnifier))

</div>

In [ ]:
studies = mg.studies(
    search=...
)
iss_dust_studies = studies.bulk_fetch()
iss_dust_studies.enrich_details()

**Try to complete the cell above. If you get too stuck, unfold the cell below to see a solution.**

In [ ]:
studies = mg.studies(
    search="international space station vacuum cleaner"
)
iss_dust_studies = studies.bulk_fetch()
iss_dust_studies.enrich_details()

As before, there should really only be one relevant study, let's check:

In [ ]:
print(f"There are {iss_dust_studies.count} matching studies")

iss_study = iss_dust_studies.first()
mgazine = iss_dust_studies.datasets
downloads = mgazine.downloads_df()
downloads

#### 1.5.2 Use pandas

Now we have a regular `pandas` dataframe representing the downloads.

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">

☑️  Use the `pipeline_version` column to [select the rows](https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html#how-do-i-filter-specific-rows-from-a-dataframe) where `pipeline_version` is `v6`

</div>

In [ ]:
v6_downloads = downloads....

v6_downloads

**Try to complete the cell above. If you get too stuck, unfold the cell below to see a solution.**

In [ ]:
v6_downloads = downloads[downloads.pipeline_version == 'v6']

v6_downloads

Now, we can just [follow the MGnipy instructions](https://mgnipy.mgnify.org/notebooks/getting-started/4_downloading_data.html#accessing-the-mgazine-of-datasets) to load e.g. the SSU taxonomic summary (which tells us what bacteria (mostly) are in the samples of this study):

In [ ]:
ssu = mgazine['Summary of SILVA-SSU taxonomies']
ssu.load()
ssu_df = ssu.to_pandas()
ssu_df


Now we have some MGnify data as a `pandas` `Dataframe`, and we could do any kind of interesting downstream analysis with it, like plotting or machine learning. But that is beyond the scope of this session.

---

## 2. Using the ENA API

MGnify happens to have a close relationship with the European Nucleotide Archive, where the originally deposited sequencing data can be found.

### 3.1 Finding files on ENA

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">

- ☑️ Go to the [ENA Portal API interative documentation](https://www.ebi.ac.uk/ena/portal/api/swagger-ui/index.html) and see if you can work out how to use `/filereport` endpoint to get a list of the `FASTQ` files for the first `read_run` we were looking at from MGnify (`SRR1139729`). You can use the "Try it out" button to play around until you work out the right URL...
- ☑️ Once you have it, try getting that File Report into a `pandas` dataframe. Even though we aren't using a Client Library (like MGnipy) for ENA, we can still get API results into a pandas Dataframe by just passing the URL into `read_csv`:

</div>

In [ ]:
import pandas as pd

In [ ]:
pd.read_csv(  ...   , sep='\t')

**Try to complete the cell above. If you get too stuck, unfold the cell below to see a solution.**

In [ ]:
import pandas as pd
pd.read_csv("https://www.ebi.ac.uk/ena/portal/api/filereport?result=read_run&accession=SRR1139729&limit=10&format=tsv", sep='\t')

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">

☑️ 🤔 Something to think about: if you had hundreds of these files to download (perhaps you'd queried all the read_run files in a big study), how might you do it? What would you have to consider?

</div>

### 2.2 Finding metadata on ENA/BioSamples

There are a few possible ways to get the metadata associated with a MGnify/ENA sample... 

Let's take another ISS dust sample (this time from the Russian part of the station): `SAMEA4359035`

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">
    
☑️ Have a look at [this sample's MGnify web page](https://www.ebi.ac.uk/metagenomics/samples/SAMEA4359035)

</div>

- You'll see there is a fairly long list of metadata key:value pairs present.
- These are sourced from ENA, and we could construct an API call to retrieve them (e.g. a subset):

In [ ]:
%%bash

curl -s "https://www.ebi.ac.uk/ena/portal/api/search?result=sample&query=accession%3DSAMEA4359035&fields=accession%2Cstudy_accession%2Cdescription%2Cbroad_scale_environmental_context%2Clat%2Clon&limit=10&format=json&download=false" | jq

By the way: here, we have received JSON back, rather than a tabular TSV structure.
If you wanted to read this JSON into Python, you still can! For example straight into a Pandas dataframe...

In [ ]:
pd.read_json("https://www.ebi.ac.uk/ena/portal/api/search?result=sample&query=accession%3DSAMEA4359035&fields=accession%2Cstudy_accession%2Cdescription%2Cbroad_scale_environmental_context%2Clat%2Clon&limit=10&format=json&download=false")

BUT - what about metadata that *wasn't* provided to ENA when the sequence was deposited?

--- 


## 3. Using the Europe PMC Annotations API

### 3.1 Finding primers from a paper associated with the dataset

A very common case is that a sequencing read may be deposited to ENA without including the _primers_ (short DNA sequences used during the sequencing).
For some analyses, knowing these can be quite important, and often one has to dig around in the publication associated with a dataset to find them.

The last API we're looking at can help us do this automatically...

#### 3.1.2 Find a Pubmed Article ID using a Europe PMC API

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">

- ☑️ Go to the [Europe PMC Articles API developer docs](https://europepmc.org/RestfulWebService#quick)
  - This API lets us find articles (e.g. publications) for example by searching keywords
  - ☑️ Check out the "Quick Start" section, and see if you can work out how to find the Article ID of a publication (there is only one!) associated with the `study_accession` we just got from ENA's API (`PRJEB...`)
- ☑️ Then, go to the [Europe PMC Annotations API developer docs](https://europepmc.org/AnnotationsApi#!/annotations45api45controller/getAnnotationsArticlesByIdsUsingGET)
  - This API lets us find annotations (usually results of text-mining the articles for useful terms)
  - ☑️ Check out the `get /europepmc/annotations_api/annotationsByArticleId` endpoint...
  - ☑️ Can you work out the URL to get the Annotations, for the article ID we just fetched?

</div>

And then:

<div style="
  background-color:#e6f3ff;
  border-left:6px solid #2196F3;
  padding:10px 16px;
  border-radius:6px;
  font-size:14px;
">

- ☑️ Try and write some bash (using `curl`, `jq` etc), or python (using `pandas` etc), that will make a table of all the primer sequences mentioned in the paper, that therefore may be relevant to our ISS dust sample.

</div>

  - A tip if you go for python... if you want to build a pandas dataframe from some JSON that is a bit nested in an API response, you can do things like this with `httpx`:
```python
import pandas as pd
import httpx
response = httpx.get("http://some-api/")
results = response.json()["resultList"][0]["results"]  # for example, to navigate down inside the JSON a bit
results_df = pd.json_normalize(results)
```

In [ ]:
%%bash

# curl ... | jq ... etc

In [ ]:
%%bash

SAMPLE="SAMEA4359035"

STUDY=$(
    curl -s "https://www.ebi.ac.uk/ena/portal/api/search?result=sample&query=accession%3D${SAMPLE}&fields=accession%2Cstudy_accession%2Cdescription%2Cbroad_scale_environmental_context%2Clat%2Clon&limit=10&format=json&download=false" |
    jq -r '.[0].study_accession'
)

ARTICLE_ID=$(
    curl -s "https://www.ebi.ac.uk/europepmc/webservices/rest/search?query=${STUDY}&format=json" |
    jq -r '.resultList.result[0].pmid'
)


curl -s "https://www.ebi.ac.uk/europepmc/annotations_api/annotationsByArticleIds?articleIds=MED:${ARTICLE_ID}" |
jq -r '.[0].annotations[] | select(.type=="Primer") | .exact'

In [ ]:
import pandas as pd
import httpx

sample="SAMEA4359035"

...

In [ ]:
import pandas as pd
import httpx

sample="SAMEA4359035"


sample_ena_df = pd.read_json(f"https://www.ebi.ac.uk/ena/portal/api/search?result=sample&query=accession%3D{sample}&fields=accession%2Cstudy_accession%2Cdescription%2Cbroad_scale_environmental_context%2Clat%2Clon&limit=10&format=json")
study_accession = sample_ena_df.study_accession[0]

# this one is a bit fiddly 
articles_epmc_response = httpx.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={study_accession}&format=json")
pubmed_id = articles_epmc_response.json()["resultList"]["result"][0]["pmid"]

annotations = httpx.get(f"https://www.ebi.ac.uk/europepmc/annotations_api/annotationsByArticleIds?articleIds=MED:{pubmed_id}")
annotations_df = pd.json_normalize(annotations.json()[0]["annotations"])
annotations_df = annotations_df[annotations_df.type=="Primer"]
annotations_df.exact

---

That's it for now!

EMBL-EBI has APIs for almost all of its services. They use various formats, but all have documentation. 
Some, like MGnify, provide Python client libraries too.

LLMs and coding agents also tend to be extremely good at working out how to use these APIs, especially if you point them at the documentation and/or schema!

Web APIs like this are far from unique to EMBL-EBI, and a huge number of scientific and other services on the web either use APIs internally, or present them publicly for general use.

Creating your own APIs is a topic for another day, but if you develop and want to distribute and provide access to a large database of interesting data/result, or provide a novel web service like an interesting search algorithm, developing an API for it is a great way to make it accessible to more use cases.

Look into Fast API and Django Ninja as too good places to start in the Python world.